In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import scipy.stats as stats
from pandas.plotting import register_matplotlib_converters

register_matplotlib_converters()
%matplotlib inline
pd.options.display.float_format = '{:,.2f}'.format

pio.renderers.default = "browser"

years = mdates.YearLocator()
months = mdates.MonthLocator()
years_fmt = mdates.DateFormatter('%Y')

In [ ]:
df_yearly = pd.read_csv('annual_deaths_by_clinic.csv')
df_monthly = pd.read_csv('monthly_deaths.csv', parse_dates=['date'])

print(f"Yearly Shape: {df_yearly.shape}")
print(f"Monthly Shape: {df_monthly.shape}")
print(f"Any yearly duplicates? {df_yearly.duplicated().values.any()}")
print(f"Any monthly duplicates? {df_monthly.duplicated().values.any()}\n")

prob = df_yearly.deaths.sum() / df_yearly.births.sum() * 100
print(f"Chances of dying in the 1840s in Vienna: {prob:.3f}%\n")

display(df_yearly.describe())
display(df_monthly.describe())

In [ ]:
plt.figure(figsize=(14, 8), dpi=200)
plt.title('Total Number of Monthly Births and Deaths', fontsize=18)
plt.yticks(fontsize=14)
plt.xticks(fontsize=14, rotation=45)

ax1 = plt.gca()
ax2 = ax1.twinx()

ax1.set_ylabel('Births', color='skyblue', fontsize=18)
ax2.set_ylabel('Deaths', color='crimson', fontsize=18)

# Format the time x-axis
ax1.set_xlim([df_monthly.date.min(), df_monthly.date.max()])
ax1.xaxis.set_major_locator(years)
ax1.xaxis.set_major_formatter(years_fmt)
ax1.xaxis.set_minor_locator(months)
ax1.grid(color='grey', linestyle='--')

ax1.plot(df_monthly.date, df_monthly.births, color='skyblue', linewidth=3)
ax2.plot(df_monthly.date, df_monthly.deaths, color='crimson', linewidth=2, linestyle='--')
plt.show()

In [ ]:
df_yearly['pct_deaths'] = df_yearly.deaths / df_yearly.births

clinic_1 = df_yearly[df_yearly.clinic == 'clinic 1']
avg_c1 = clinic_1.deaths.sum() / clinic_1.births.sum() * 100
print(f'Average death rate in clinic 1 is {avg_c1:.3f}%.')

clinic_2 = df_yearly[df_yearly.clinic == 'clinic 2']
avg_c2 = clinic_2.deaths.sum() / clinic_2.births.sum() * 100
print(f'Average death rate in clinic 2 is {avg_c2:.3f}%.')

line_births = px.line(df_yearly, x='year', y='births', color='clinic', title='Total Yearly Births by Clinic')
line_births.show()

line_deaths = px.line(df_yearly, x='year', y='deaths', color='clinic', title='Total Yearly Deaths by Clinic')
line_deaths.show()

line_pct = px.line(df_yearly, x='year', y='pct_deaths', color='clinic', title='Proportion of Yearly Deaths by Clinic')
line_pct.show()

In [ ]:
handwashing_start = pd.to_datetime('1847-06-01')
df_monthly['pct_deaths'] = df_monthly.deaths / df_monthly.births

before_washing = df_monthly[df_monthly.date < handwashing_start]
after_washing = df_monthly[df_monthly.date >= handwashing_start]

bw_rate = before_washing.deaths.sum() / before_washing.births.sum() * 100
aw_rate = after_washing.deaths.sum() / after_washing.births.sum() * 100
print(f'Average death rate before 1847 was {bw_rate:.4f}%')
print(f'Average death rate AFTER 1847 was {aw_rate:.3f}%')

In [ ]:
roll_df = before_washing.set_index('date').rolling(window=6).mean()

plt.figure(figsize=(14, 8), dpi=200)
plt.title('Percentage of Monthly Deaths over Time', fontsize=18)
plt.yticks(fontsize=14)
plt.xticks(fontsize=14, rotation=45)
plt.ylabel('Percentage of Deaths', color='crimson', fontsize=18)

ax = plt.gca()
ax.xaxis.set_major_locator(years)
ax.xaxis.set_major_formatter(years_fmt)
ax.xaxis.set_minor_locator(months)
ax.set_xlim([df_monthly.date.min(), df_monthly.date.max()])
plt.grid(color='grey', linestyle='--')

ma_line, = plt.plot(roll_df.index, roll_df.pct_deaths, color='crimson', linewidth=3, linestyle='--', label='6m Moving Average')
bw_line, = plt.plot(before_washing.date, before_washing.pct_deaths, color='black', linewidth=1, linestyle='--', label='Before Handwashing')
aw_line, = plt.plot(after_washing.date, after_washing.pct_deaths, color='skyblue', linewidth=3, marker='o', label='After Handwashing')

plt.legend(handles=[ma_line, bw_line, aw_line], fontsize=18)
plt.show()

In [ ]:
avg_prob_before = before_washing.pct_deaths.mean() * 100
avg_prob_after = after_washing.pct_deaths.mean() * 100
mean_diff = avg_prob_before - avg_prob_after
times = avg_prob_before / avg_prob_after

print(f'Chance of death during childbirth before handwashing: {avg_prob_before:.3f}%.')
print(f'Chance of death during childbirth AFTER handwashing: {avg_prob_after:.3f}%.')
print(f'Handwashing reduced the monthly proportion of deaths by {mean_diff:.3f}%!')
print(f'This is a {times:.2f}x improvement!')

In [ ]:
df_monthly['washing_hands'] = np.where(df_monthly.date < handwashing_start, 'No', 'Yes')

box = px.box(
    df_monthly, 
    x='washing_hands', 
    y='pct_deaths', 
    color='washing_hands', 
    title='How Have the Stats Changed with Handwashing?'
)
box.update_layout(xaxis_title='Washing Hands?', yaxis_title='Percentage of Monthly Deaths')
box.show()

hist = px.histogram(
    df_monthly, 
    x='pct_deaths', 
    color='washing_hands', 
    nbins=30, 
    opacity=0.6, 
    barmode='overlay', 
    histnorm='percent', 
    marginal='box'
)
hist.update_layout(xaxis_title='Proportion of Monthly Deaths', yaxis_title='Count')
hist.show()

In [ ]:
plt.figure(dpi=200)
sns.kdeplot(before_washing.pct_deaths, fill=True, clip=(0, 1), label='Before Handwashing')
sns.kdeplot(after_washing.pct_deaths, fill=True, clip=(0, 1), label='After Handwashing')

plt.title('Est. Distribution of Monthly Death Rate Before and After Handwashing')
plt.xlim(0, 0.40)
plt.legend()
plt.show()

In [ ]:
t_stat, p_value = stats.ttest_ind(a=before_washing.pct_deaths, b=after_washing.pct_deaths)

print(f't-statistic is {t_stat:.4f}')
print(f'p-value is {p_value:.10f}')